In [1]:
# ============================================================
# CELL 1 — Spark Configuration + Imports
# GlobalWatch: Bronze Ingestion — OpenAQ API
# ============================================================

# --- Spark Optimization Settings ---
# AQE: lets Spark dynamically optimize shuffle partitions at runtime
spark.conf.set("spark.sql.adaptive.enabled", "true")
# Coalesce small partitions after shuffle — avoids 200 tiny files
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Auto-detect and handle skewed partitions (e.g. high-volume city stations)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# --- Standard Imports ---
import requests                              # HTTP calls to OpenAQ API
from datetime import datetime, timezone      # UTC timestamp handling
from pyspark.sql import functions as F       # PySpark column functions
from pyspark.sql.types import *              # Schema type definitions

# --- Database Context ---
# Fabric encodes the lakehouse path into an internal DB name
# We let Fabric tell us its own name rather than hardcoding it
# This avoids the SCHEMA_NOT_FOUND error from name duplication
DB = spark.sql("SELECT current_database()").collect()[0][0]

# --- API Key ---
# Stored securely in Fabric Environment Spark properties
# Key: spark.openaq.api.key — never hardcoded in notebook
OPENAQ_API_KEY = spark.conf.get("spark.openaq.api.key")

print(f"Config loaded ✅")
print(f"DB context: {DB}")
print(f"API Key loaded: {'✅' if OPENAQ_API_KEY else '❌ NOT FOUND — check environment'}")

StatementMeta(, d0502c70-b149-4ccc-9a67-a7939ac0b1da, 3, Finished, Available, Finished, False)

Config loaded ✅
DB context: chimcobldhq2aprcdth62r3nc5q66q1dchinc9b2e9nmsuj5btjmorr2c5m7eobkcdk2ap32ds
API Key loaded: ✅


In [2]:
# ============================================================
# CELL 2 — Watermark Control Table
# Purpose: Track last loaded date per source
# Pattern: Incremental ingestion — only pull new data each run
# ============================================================

# Create watermark table if it doesn't exist
# USING DELTA: enables ACID transactions + time travel
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB}.watermark_control (
    source_name      STRING,      -- source identifier e.g. 'openaq_batch'
    last_loaded_date DATE,        -- date of last successful load
    last_loaded_ts   TIMESTAMP    -- full timestamp of last successful load
) USING DELTA
""")

# Seed initial watermark only if table is empty
count = spark.sql(f"""
    SELECT COUNT(*) as cnt FROM {DB}.watermark_control
""").collect()[0]['cnt']

if count == 0:
    # Start from Jan 2024 — historical backfill starting point
    spark.sql(f"""
    INSERT INTO {DB}.watermark_control VALUES
    ('openaq_batch', '2024-01-01', '2024-01-01T00:00:00'),
    ('waqi_batch',   '2024-01-01', '2024-01-01T00:00:00')
    """)
    print("Initial watermark seeded ✅")
else:
    print(f"Watermark table already has {count} rows — skipping seed")

# Display current watermark state
spark.sql(f"SELECT * FROM {DB}.watermark_control").show()
print("Watermark table ready ✅")

StatementMeta(, d0502c70-b149-4ccc-9a67-a7939ac0b1da, 4, Finished, Available, Finished, False)

Watermark table already has 2 rows — skipping seed
+------------+----------------+-------------------+
| source_name|last_loaded_date|     last_loaded_ts|
+------------+----------------+-------------------+
|openaq_batch|      2024-01-01|2024-01-01 00:00:00|
|  waqi_batch|      2024-01-01|2024-01-01 00:00:00|
+------------+----------------+-------------------+

Watermark table ready ✅


In [4]:
# ============================================================
# CELL 3 — OpenAQ API Functions + Connectivity Test
# API: OpenAQ v3 — https://api.openaq.org/v3
# Auth: X-API-Key header (key from Fabric environment)
# Free tier: 60 req/min, global coverage, 10K+ stations
# ============================================================

OPENAQ_BASE = "https://api.openaq.org/v3"

def fetch_openaq_locations(limit=50, page=1):
    """
    Fetch air quality station metadata.
    Returns station ID, name, country, coordinates.
    Used to discover which stations to pull measurements from.
    """
    url = f"{OPENAQ_BASE}/locations"
    params = {
        "limit": limit,   # stations per page (max 1000)
        "page": page      # pagination — 1-indexed
    }
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY   # required for v3
    }
    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        if r.status_code == 200:
            return r.json()
        else:
            print(f"  API error {r.status_code}: {r.text[:100]}")
            return None
    except Exception as e:
        print(f"  Request failed: {e}")
        return None

def fetch_openaq_measurements(location_id, limit=20):
    """
    Fetch latest pollutant measurements for a specific station.
    Returns PM2.5, PM10, NO2, CO, O3 readings with timestamps.
    """
    url = f"{OPENAQ_BASE}/locations/{location_id}/measurements"
    params = {"limit": limit}
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY
    }
    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        if r.status_code == 200:
            return r.json()
        return None
    except Exception as e:
        print(f"  Measurement fetch failed for location {location_id}: {e}")
        return None

# --- Connectivity Test ---
print("Testing OpenAQ API connectivity...")
test = fetch_openaq_locations(limit=3)
if test:
    total_found = test['meta']['found']
    print(f"API connected ✅ — {total_found} stations globally")
    print("Sample stations:")
    for loc in test['results']:
        code = loc.get('country', {}).get('code', '??')
        name = loc.get('name', 'N/A')
        country = loc.get('country', {}).get('name', 'N/A')
        print(f"  → [{code}] {name} | {country}")
else:
    print("❌ API connection failed — check API key in environment settings")

StatementMeta(, d0502c70-b149-4ccc-9a67-a7939ac0b1da, 6, Finished, Available, Finished, False)

Testing OpenAQ API connectivity...
API connected ✅ — >3 stations globally
Sample stations:
  → [GH] NMA - Nima | Ghana
  → [GH] NMT - Nima | Ghana
  → [GH] JTA - Jamestown | Ghana


In [11]:
# ============================================================
# CELL 4 — Fixed: Fetch Measurements + Write to Bronze Delta
# Fix: Build sensor map from location data
#      parameter + unit extracted from sensors[] array
#      city falls back to country name if locality is None
#      Rate limit: 1 sec delay between requests
# ============================================================

import time

schema = StructType([
    StructField("location_id",   IntegerType()),
    StructField("location_name", StringType()),
    StructField("city",          StringType()),
    StructField("country_code",  StringType()),
    StructField("country_name",  StringType()),
    StructField("latitude",      DoubleType()),
    StructField("longitude",     DoubleType()),
    StructField("parameter",     StringType()),
    StructField("value",         DoubleType()),
    StructField("unit",          StringType()),
    StructField("reading_ts",    TimestampType()),
    StructField("ingestion_ts",  TimestampType()),
    StructField("source_system", StringType())
])

def fetch_location_sensors(location_id):
    """
    GET /v3/locations/{id}/sensors
    Returns all sensors at a location with parameter metadata.
    Used to build sensor_id → parameter/unit mapping.
    """
    url = f"{OPENAQ_BASE}/locations/{location_id}/sensors"
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY
    }
    try:
        r = requests.get(url, headers=headers, timeout=30)
        return r.json() if r.status_code == 200 else None
    except Exception as e:
        print(f"  Sensor fetch error: {e}")
        return None

def fetch_sensor_latest(sensor_id):
    """
    GET /v3/sensors/{id}/measurements/hourly
    Returns latest hourly measurement for a specific sensor.
    This is the correct v3 path per OpenAQ docs.
    """
    url = f"{OPENAQ_BASE}/sensors/{sensor_id}/measurements/hourly"
    params = {"limit": 1}
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY
    }
    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        return r.json() if r.status_code == 200 else None
    except Exception as e:
        print(f"  Sensor measurement error: {e}")
        return None

# --- Main Fetch Loop ---
records = []
ingestion_ts = datetime.now(timezone.utc)
skipped = 0
rate_limited = 0

for page in range(1, 4):
    print(f"\nFetching locations page {page}/3...")
    locations_data = fetch_openaq_locations(limit=50, page=page)

    if not locations_data or not locations_data.get('results'):
        print(f"  No data on page {page} — stopping")
        break

    for loc in locations_data['results']:
        location_id   = loc.get('id')
        location_name = loc.get('name', '')

        # city: use locality, fall back to country name
        city = loc.get('locality') or loc.get('country', {}).get('name', '')

        # country
        country      = loc.get('country', {}) if isinstance(loc.get('country'), dict) else {}
        country_code = country.get('code', '')
        country_name = country.get('name', '')

        # coordinates
        coords    = loc.get('coordinates', {}) if isinstance(loc.get('coordinates'), dict) else {}
        latitude  = coords.get('latitude')
        longitude = coords.get('longitude')

        # Build sensor map from location sensors[] array
        # sensors[].parameter.name → pollutant name
        # sensors[].parameter.units → unit
        sensors = loc.get('sensors', [])
        if not sensors:
            skipped += 1
            continue

        for sensor in sensors:
            sensor_id = sensor.get('id')
            param     = sensor.get('parameter', {})
            param_name = param.get('name', '') if isinstance(param, dict) else ''
            param_unit = param.get('units', '') if isinstance(param, dict) else ''

            # Only fetch key pollutants
            if param_name not in ('pm25', 'pm10', 'no2', 'co', 'o3'):
                continue

            # Rate limit protection — 1 request per second
            time.sleep(1)

            # Fetch latest measurement for this sensor
            measurement = fetch_sensor_latest(sensor_id)
            if not measurement or not measurement.get('results'):
                continue

            for m in measurement['results']:
                try:
                    # Parse timestamp
                    period = m.get('period', {})
                    ts_str = period.get('datetimeFrom', {}).get('utc', '') \
                             if isinstance(period, dict) else ''
                    reading_ts = datetime.fromisoformat(
                        ts_str.replace('Z', '+00:00')
                    ) if ts_str else ingestion_ts

                    # Coverage value
                    coverage = m.get('coverage', {})
                    value = float(coverage.get('avg', 0) or
                                  m.get('value', 0) or 0)

                    records.append((
                        location_id,
                        location_name,
                        city,
                        country_code,
                        country_name,
                        latitude,
                        longitude,
                        param_name,   # correctly parsed from sensor metadata
                        value,
                        param_unit,   # correctly parsed from sensor metadata
                        reading_ts,
                        ingestion_ts,
                        'openaq_v3'
                    ))

                except Exception as e:
                    print(f"  Skipped record: {e}")
                    continue

        # Delay between locations to avoid 429
        time.sleep(0.5)

print(f"\nFetch complete:")
print(f"  Total records : {len(records)}")
print(f"  Stations skipped: {skipped}")

# --- Write to Bronze Delta ---
if records:
    df = spark.createDataFrame(records, schema=schema)

    df = df \
        .withColumn("ingestion_date",
                    F.to_date("ingestion_ts")) \
        .withColumn("year_month",
                    F.date_format("ingestion_ts", "yyyy-MM"))

    df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("ingestion_date") \
        .saveAsTable(f"{DB}.raw_openaq_readings")

    print(f"Written {df.count()} rows → Bronze ✅")

    print("\nSample data:")
    df.select(
        "country_code", "country_name", "city",
        "parameter", "value", "unit", "reading_ts"
    ).show(10, truncate=False)

else:
    print("❌ No records written")
    # Debug: print first sensor response
    test_loc = fetch_openaq_locations(limit=1)
    if test_loc and test_loc.get('results'):
        test_sensors = test_loc['results'][0].get('sensors', [])
        if test_sensors:
            sid = test_sensors[0].get('id')
            print(f"Debug sensor {sid} response:")
            print(fetch_sensor_latest(sid))

StatementMeta(, d0502c70-b149-4ccc-9a67-a7939ac0b1da, 13, Finished, Available, Finished, False)


Fetching locations page 1/3...

Fetching locations page 2/3...

Fetching locations page 3/3...

Fetch complete:
  Total records : 368
  Stations skipped: 0
Written 368 rows → Bronze ✅

Sample data:
+------------+------------+-----+---------+------+-----+-------------------+
|country_code|country_name|city |parameter|value |unit |reading_ts         |
+------------+------------+-----+---------+------+-----+-------------------+
|IN          |India       |India|no2      |11.3  |µg/m³|2016-11-02 17:30:00|
|IN          |India       |India|pm25     |300.0 |µg/m³|2016-11-02 17:30:00|
|IN          |India       |India|co       |1.65  |ppb  |2025-02-18 19:30:00|
|IN          |India       |India|co       |6850.0|µg/m³|2016-02-05 14:30:00|
|IN          |India       |India|no2      |148.0 |µg/m³|2016-02-05 14:30:00|
|IN          |India       |India|no2      |41.3  |ppb  |2025-02-18 19:30:00|
|IN          |India       |India|o3       |16.0  |µg/m³|2025-02-18 19:30:00|
|IN          |India       |Indi

In [12]:
# ============================================================
# CELL 5 — Validation + Watermark Update
# ============================================================

# --- Row Count ---
total = spark.sql(f"""
    SELECT COUNT(*) as total FROM {DB}.raw_openaq_readings
""").collect()[0]['total']
print(f"Total rows in Bronze: {total}")
assert total > 0, "❌ VALIDATION FAILED: Bronze table is empty!"

# --- Data Quality Summary ---
print("\nTop 10 by reading count:")
spark.sql(f"""
    SELECT
        country_code,
        country_name,
        parameter,
        ROUND(AVG(value), 2)  AS avg_value,
        ROUND(MIN(value), 2)  AS min_value,
        ROUND(MAX(value), 2)  AS max_value,
        COUNT(*)              AS readings
    FROM {DB}.raw_openaq_readings
    GROUP BY country_code, country_name, parameter
    ORDER BY readings DESC
    LIMIT 10
""").show(truncate=False)

# --- Partition Check ---
print("Partitions written:")
spark.sql(f"""
    SELECT ingestion_date, COUNT(*) as rows
    FROM {DB}.raw_openaq_readings
    GROUP BY ingestion_date
""").show()

# --- Update Watermark ---
spark.sql(f"""
    UPDATE {DB}.watermark_control
    SET last_loaded_ts   = current_timestamp(),
        last_loaded_date = current_date()
    WHERE source_name = 'openaq_batch'
""")

print("Watermark updated ✅")
print("Bronze ingestion complete ✅")
print("Next: Run 04_silver_transform notebook")

StatementMeta(, d0502c70-b149-4ccc-9a67-a7939ac0b1da, 14, Finished, Available, Finished, False)

Total rows in Bronze: 703

Top 10 by reading count:
+------------+--------------+---------+---------+---------+---------+--------+
|country_code|country_name  |parameter|avg_value|min_value|max_value|readings|
+------------+--------------+---------+---------+---------+---------+--------+
|NL          |Netherlands   |         |-145.57  |-999.0   |934.0    |117     |
|CL          |Chile         |         |140.67   |0.0      |2906.65  |81      |
|IN          |India         |         |189.29   |0.02     |4300.0   |41      |
|MN          |Mongolia      |         |92.44    |2.0      |821.0    |39      |
|US          |United States |o3       |0.03     |0.01     |0.05     |36      |
|NL          |Netherlands   |pm10     |14.42    |6.46     |23.0     |24      |
|NL          |Netherlands   |no2      |7.91     |0.0      |19.2     |23      |
|GB          |United Kingdom|         |19.12    |-1.0     |129.57   |21      |
|NL          |Netherlands   |pm25     |7.6      |2.63     |16.6     |20      |
